# Phase 3: Campaign Targeting (segment effects)
Phase 2 showed that, *on average*, both emails raise the visit rate, and the men's email more than the women's. An average can hide differences between customers: some groups may respond better to the women's email (for example customers who only bought women's items before), and some may not respond at all, or even react badly, when contacted.

We look for these differences in two ways:
1. **This notebook: segment effects.** We split the customers into segments using one feature at a time (for example recency in three bands) and estimate the lift of each email inside each segment. In causal-inference terms, each of these lifts is the CATE (conditional average treatment effect) of that segment.
2. **Phase 4: an uplift model.** A meta-learner takes all the features together and predicts the lift for each individual customer.

**Choices made before looking at the results:**
- **Outcome:** `visit`. Conversion is too rare to split into segments (about 1% of customers buy, so most segment × group cells have fewer than 100 buyers), and in phase 2 spend was already noisy on the full sample.
- **Segments:** six features, one at a time, with three bands for the numerical ones (see below). The overlap between features is left to phase 4.
- **Tests:** two-proportion z-tests within segments and interaction tests between segments, Holm-corrected for the number of tests.

In [1]:
import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

from digital_sales.experiment import proportion_test, values
from digital_sales.paths import DB_PATH, SQL_DIR

# Segments
We define one segment dimension per pre-campaign feature:
- **Recency** and **past-year spend** (`history`) are numerical, so we split each into three bands with about the same number of customers (tertiles, from `pd.qcut` below). Equal-sized bands give every segment a similar precision. The number of bands is arbitrary, but it is fixed before looking at the lifts.
- **Buyer type** combines the `mens` and `womens` flags: men's items only, women's items only, or both. Every customer bought at least one of the two.
- **Channel**, **new vs existing customer** (`newbie`) and **area** (`zip_code`) keep their own categories.

In [2]:
## choosing the bins 
# to choose the bins for numerical variables we split them so that the groups have more or less the same counts
# we arbitrarily split into 3 groups
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    df = con.table('analysis_base').df()

pd.qcut(df['recency'].astype(float), q=3).value_counts() #0-3-8 
pd.qcut(df['history'].astype(float), q=3).value_counts() #0 - 92 - 255
pass

In [3]:
# create the binned columns (segment definitions in sql/03_segments.sql)
SEGMENTS_SQL = (SQL_DIR / "03_segments.sql").read_text(encoding="utf-8")
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    seg = con.sql(SEGMENTS_SQL).df()

DIMENSIONS = {
    "recency_band": ("Recency", ["1-3 months", "4-8 months", "9-12 months"]),
    "history_band": ("Past-year spend", ["Under $92", "$92-254", "$255+"]),
    "buyer_type":   ("Buyer type", ["Men's only", "Women's only", "Both"]),
    "channel":      ("Channel", ["Phone", "Web", "Multichannel"]),
    "newbie":       ("Customer", ["New", "Existing"]),
    "zip_code":     ("Area", ["Urban", "Suburban", "Rural"]),
}

### Balance within segments
Randomisation should split every segment into roughly equal thirds across the three groups, as it did for the whole sample in phase 2. The table confirms it, and it shows how many customers each segment lift rests on: the smallest segment ("Both" buyers) still has about 2,100 customers per group.

In [4]:
# verify the segments are uniformly distributed among treatments
sizes = pd.concat(
    {label: pd.crosstab(seg[dim], seg["treatment"]).reindex(order)
     for dim, (label, order) in DIMENSIONS.items()},
    names=["Dimension", "Segment"],
)[["no_email", "womens_email", "mens_email"]]
sizes

treatment                     no_email  womens_email  mens_email
Dimension       Segment                                         
Recency         1-3 months        7438          7486        7469
                4-8 months        7339          7269        7157
                9-12 months       6529          6632        6681
Past-year spend Under $92         7112          7149        7219
                $92-254           7116          7085        7024
                $255+             7078          7153        7064
Buyer type      Men's only        9638          9622        9558
                Women's only      9519          9647        9568
                Both              2149          2118        2181
Channel         Phone             9327          9454        9240
                Web               9373          9354        9490
                Multichannel      2606          2579        2577
Customer        New              10695         10763       10686
                Existing         10611         10624       10621
Area            Urban             8542          8556        8563
                Suburban          9625          9650        9501
                Rural             3139          3181        3243

In [5]:
DIMENSIONS

{'recency_band': ('Recency', ['1-3 months', '4-8 months', '9-12 months']),
 'history_band': ('Past-year spend', ['Under $92', '$92-254', '$255+']),
 'buyer_type': ('Buyer type', ["Men's only", "Women's only", 'Both']),
 'channel': ('Channel', ['Phone', 'Web', 'Multichannel']),
 'newbie': ('Customer', ['New', 'Existing']),
 'zip_code': ('Area', ['Urban', 'Suburban', 'Rural'])}

# Lift per segment
For each segment we compare each email with the no-email group, using the same two-proportion z-test as in phase 2 (`proportion_test`) on that segment's customers only. Each result is the visit lift of that email *within* the segment.

These 34 confidence intervals are descriptive, so they are not corrected for multiple testing. Whether the lift really differs between segments is tested formally in the next section.

In [6]:
## We choose the visit as primary outcome variable, we check whether segments affect the visit rate of customers
EMAILS = ["mens_email", "womens_email"]

rows = []
for dim, (label, order) in DIMENSIONS.items():      # 1. each dimension (recency, history, ...)
    for segment in order:                           # 2. each segment within it ("1-3 months", ...)
        sub = seg[seg[dim] == segment]              #    keep only this segment's customers
        for email in EMAILS:                        # 3. each treatment except control
            result = proportion_test(
                values(sub, "visit", email),        #    y_t: 0/1 visits of emailed customers in the segment
                values(sub, "visit", "no_email"),   #    y_r: 0/1 visits of non-emailed customers in the segment
            )
            rows.append({"dimension": label, "segment": segment, "email": email,
                         **result})                 #    ** merges proportion_test's dict into this row
seg_lifts = pd.DataFrame(rows)                      # one row per segment × email: 17 × 2 = 34 rows

In [7]:
## compact the previous dataframe into a readable table
def fmt_lift(lift, low, high):
    return f"{lift * 100:+.1f} pp ({low * 100:.1f} to {high * 100:.1f})"

seg_lifts["lift_text"] = [fmt_lift(r.abs_lift, r.abs_ci_low, r.abs_ci_high) for r in seg_lifts.itertuples()]
ROWS = [(label, s) for label, order in DIMENSIONS.values() for s in order]

seg_table = (seg_lifts.pivot(index=["dimension", "segment"], columns="email", values="lift_text")
             .reindex(ROWS)[EMAILS])
baseline = seg_lifts.groupby(["dimension", "segment"])["reference_value"].first()
seg_table.insert(0, "no_email_visit_rate", baseline.map("{:.1%}".format))
seg_table.columns = ["No-email visit rate", "Men's email lift (95% CI)", "Women's email lift (95% CI)"]
seg_table

No-email visit rate Men's email lift (95% CI)  \
dimension       segment                                                      
Recency         1-3 months                 13.8%      +8.5 pp (7.3 to 9.7)   
                4-8 months                 10.2%      +7.0 pp (5.9 to 8.1)   
                9-12 months                 7.5%      +7.4 pp (6.4 to 8.5)   
Past-year spend Under $92                   8.4%      +6.8 pp (5.8 to 7.9)   
                $92-254                     9.9%      +7.4 pp (6.3 to 8.5)   
                $255+                      13.6%     +8.8 pp (7.6 to 10.1)   
Buyer type      Men's only                 10.0%      +6.9 pp (6.0 to 7.9)   
                Women's only                9.6%      +7.1 pp (6.1 to 8.0)   
                Both                       18.1%   +13.4 pp (10.9 to 16.0)   
Channel         Phone                       8.7%      +7.6 pp (6.6 to 8.5)   
                Web                        11.9%      +7.6 pp (6.5 to 8.6)   
                Multichannel               12.9%     +8.3 pp (6.3 to 10.3)   
Customer        New                         7.9%      +7.5 pp (6.7 to 8.4)   
                Existing                   13.4%      +7.8 pp (6.8 to 8.8)   
Area            Urban                       9.7%      +8.0 pp (6.9 to 9.0)   
                Suburban                    9.9%      +7.4 pp (6.5 to 8.4)   
                Rural                      15.4%      +7.3 pp (5.4 to 9.3)   

                             Women's email lift (95% CI)  
dimension       segment                                   
Recency         1-3 months          +4.4 pp (3.2 to 5.6)  
                4-8 months          +4.3 pp (3.2 to 5.4)  
                9-12 months         +4.9 pp (3.9 to 6.0)  
Past-year spend Under $92           +4.8 pp (3.8 to 5.8)  
                $92-254             +3.8 pp (2.8 to 4.9)  
                $255+               +4.9 pp (3.7 to 6.1)  
Buyer type      Men's only          +1.1 pp (0.2 to 2.0)  
                Women's only        +7.4 pp (6.4 to 8.4)  
                Both                +7.1 pp (4.6 to 9.6)  
Channel         Phone               +4.5 pp (3.6 to 5.4)  
                Web                 +4.6 pp (3.6 to 5.6)  
                Multichannel        +4.7 pp (2.8 to 6.7)  
Customer        New                 +5.0 pp (4.2 to 5.9)  
                Existing            +4.0 pp (3.0 to 5.0)  
Area            Urban               +4.7 pp (3.7 to 5.7)  
                Suburban            +4.9 pp (4.0 to 5.8)  
                Rural               +2.9 pp (1.1 to 4.8)

The chart shows the same table: each bar is the extra visit rate (in percentage points) that an email produces compared with no email, with its 95% CI. The "All" group on the left is the overall lift from phase 2 (men's +7.7 pp, women's +4.5 pp), the reference every segment is compared with.

In [8]:
## plot the table 

# "All customers" group on the left: the reference every segment is compared with
overall = pd.DataFrame([
    {"dimension": "Customers", "segment": "All", "email": email,
     **proportion_test(values(seg, "visit", email), values(seg, "visit", "no_email"))}
    for email in EMAILS
])
plot_df = pd.concat([overall, seg_lifts], ignore_index=True)
plot_df["segment"] = plot_df["segment"].astype(str)

EMAIL_STYLE = {                                   # email -> (legend name, colour)
    "mens_email": ("Men's email", "#2a78d6"),
    "womens_email": ("Women's email", "#eb6834"),
}

def wrap(label):
    """Break long axis labels onto two lines so neighbouring labels don't collide."""
    return label.replace(" ", "<br>", 1) if len(label) > 8 else label

fig = go.Figure()
for email, (name, color) in EMAIL_STYLE.items():
    d = plot_df[plot_df["email"] == email]
    lift, low, high = (d[c].to_numpy() * 100 for c in ("abs_lift", "abs_ci_low", "abs_ci_high"))
    fig.add_trace(go.Bar(
        name=name,
        x=[d["dimension"], d["segment"].map(wrap)],   # two-level axis: dimension, then segment
        y=lift,
        marker=dict(color=color, cornerradius=4),
        error_y=dict(type="data", symmetric=False, array=high - lift, arrayminus=lift - low,
                     color="#52514e", thickness=1.5, width=3),
        hovertext=d["dimension"] + ": " + d["segment"],
        customdata=np.column_stack([low, high, d["treated_value"] * 100, d["reference_value"] * 100]),
        hovertemplate=(
            "<b>%{hovertext}</b><br>" + name + ": %{y:+.1f} pp"
            "<br>95% CI %{customdata[0]:.1f} to %{customdata[1]:.1f}"
            "<br>Visit rate %{customdata[2]:.1f}% vs %{customdata[3]:.1f}% with no email"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    title="Extra visits from each email, by customer segment",
    barmode="group", bargap=0.35, bargroupgap=0.08,
    template="plotly_white", height=460,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=10, r=10, t=90, b=10),
)
fig.update_yaxes(title_text="Visit lift vs no email (percentage points)",
                 zeroline=True, zerolinecolor="#9b9a95", zerolinewidth=1)
fig.update_xaxes(tickangle=0, tickfont=dict(size=11))
fig.show()

# Statistical Significance
The chart suggests that the men's email beats the women's email in almost every segment, and that the lifts look similar across most segments. These are two different questions, and they need two different tests:

1. **Within each segment, is the men's email better than the women's?** A two-proportion z-test of men's vs women's email inside each segment (17 tests, Holm-corrected). This is the next cell, which also defines two helper functions used for question 2.
2. **Does the effect of an email change between the segments of a feature?** An interaction test, in the cell after.

The two questions can give different answers. If the men's email beats the women's by about 3 pp in *every* segment, the answer to question 1 is "yes" everywhere, but the answer to question 2 is "no", because the gap is the same everywhere. Only question 2 tells us whether a feature is worth using for targeting.

In [9]:
def two_groups(treated, reference):
    "Add the treated dummy variable as 0/1 (reference/treatment)"
    data = seg[seg["treatment"].isin([treated, reference])].copy()
    data["treated"] = np.where(data["treatment"] == reference, 0, 1)
    return data

def interaction_wald(fit, dim):
    """Joint Wald test that every treated x segment term of one dimension is zero."""
    names = [n for n in fit.params.index if n.startswith(f"treated:C({dim})")]
    R = np.zeros((len(names), len(fit.params)))
    for i, name in enumerate(names):
        R[i, fit.params.index.get_loc(name)] = 1
    wald = fit.wald_test(R, use_f=False, scalar=True)
    return float(wald.statistic), len(names), float(wald.pvalue)

rows = []
for dim, (label, order) in DIMENSIONS.items():
    for segment in order:
        sub = seg[seg[dim] == segment]
        result = proportion_test(values(sub, "visit", "mens_email"), values(sub, "visit", "womens_email"))
        rows.append({"dimension": label, "segment": segment, **result})
head_to_head = pd.DataFrame(rows)
head_to_head["p_holm"] = multipletests(head_to_head["p_value"], method="holm")[1]
head_to_head.round(4)

,dimension,segment,treated_value,reference_value,abs_lift,abs_ci_low,abs_ci_high,rel_lift,rel_ci_low,rel_ci_high,p_value,p_holm
0,Recency,1-3 months,0.2228,0.1818,0.0410,0.0281,0.0538,0.2254,0.1494,0.3065,0.0000,0.0000
1,Recency,4-8 months,0.1724,0.1451,0.0273,0.0154,0.0392,0.1880,0.1017,0.2810,0.0000,0.0000
2,Recency,9-12 months,0.1491,0.1239,0.0251,0.0135,0.0368,0.2028,0.1038,0.3107,0.0000,0.0001
3,Past-year spend,Under $92,0.1521,0.1319,0.0202,0.0088,0.0316,0.1531,0.0637,0.2499,0.0005,0.0016
4,Past-year spend,$92-254,0.1727,0.1373,0.0354,0.0234,0.0473,0.2575,0.1636,0.3590,0.0000,0.0000
5,Past-year spend,$255+,0.2241,0.1848,0.0393,0.0260,0.0525,0.2125,0.1360,0.2942,0.0000,0.0000
6,Buyer type,Men's only,0.1690,0.1109,0.0581,0.0483,0.0679,0.5237,0.4179,0.6374,0.0000,0.0000
7,Buyer type,Women's only,0.1664,0.1698,-0.0034,-0.0140,0.0072,-0.0201,-0.0798,0.0436,0.5279,0.5279
8,Buyer type,Both,0.3150,0.2517,0.0633,0.0365,0.0902,0.2517,0.1371,0.3779,0.0000,0.0000
9,Channel,Phone,0.1628,0.1318,0.0310,0.0208,0.0411,0.2350,0.1522,0.3238,0.0000,0.0000


### Interaction test: is the effect the same in every segment of a feature?
For each comparison (each email vs no email, and men's vs women's email) and each feature, we fit a **linear probability model**: an ordinary least squares regression on the 0/1 `visit` outcome, so every coefficient is in percentage points.

`visit ~ treated * C(dim)`

- `treated` is a 0/1 dummy: 1 for the first group of the pair (for example the men's email) and 0 for the second (for example no email).
- `C(dim)` turns the feature into categories, with the first segment as the reference.
- The `treated` coefficient is the effect in the reference segment, and each `treated:C(dim)[T.segment]` term is **how much the effect in that segment differs** from the reference segment.

A joint Wald test checks whether all the `treated:C(dim)` terms are zero, which means the effect is the same in every segment of that feature. The standard errors are robust (HC2), because the variance of a 0/1 outcome, p(1−p), differs between segments. The 18 tests (3 comparisons × 6 features) are Holm-corrected.

**Example: recency, men's vs women's email.** The gap is 4.1 pp for customers who bought in the last 3 months, and 2.7 and 2.5 pp in the two later bands. All three gaps are clearly above zero (question 1), but they are not significantly different from each other (question 2, Holm p = 1.0). In ecological terms: a fertiliser can raise growth significantly on clay, sand and loam while the fertiliser × soil interaction is not significant, because the effect is the same on every soil.

In [10]:
COMPARISONS = [
    ("mens_email", "no_email"),
    ("womens_email", "no_email"),
    ("mens_email", "womens_email"),   # new: the men's vs women's gap
]

rows = []
for treated, reference in COMPARISONS:
    data = two_groups(treated, reference)
    for dim, (label, _) in DIMENSIONS.items():
        fit = smf.ols(f"visit ~ treated * C({dim})", data=data).fit(cov_type="HC2")
        chi2, k, p = interaction_wald(fit, dim)
        rows.append({"comparison": f"{treated} vs {reference}", "dimension": label,
                     "chi2": chi2, "df": k, "p_value": p})
het = pd.DataFrame(rows)
het["p_holm"] = multipletests(het["p_value"], method="holm")[1]

## only show the significant differences
het[het["p_holm"]<0.05].round(4)

,comparison,dimension,chi2,df,p_value,p_holm
2,mens_email vs no_email,Buyer type,23.2366,2,0.0,0.0001
8,womens_email vs no_email,Buyer type,97.3377,2,0.0,0.0000
14,mens_email vs womens_email,Buyer type,75.5513,2,0.0,0.0000


# Results

**1. Both emails raise visits in every segment.** All 34 segment lifts are above zero. Even the weakest one, the women's email on men's-only buyers (+1.1 pp, 95% CI 0.2 to 2.0), is positive, so no segment reacts badly to being contacted.

**2. For five of the six features, the effect is the same in every segment.** Across recency, past-year spend, channel, new vs existing customers and area, the men's email lift stays between +6.8 and +8.8 pp and the women's email lift between +2.9 and +5.0 pp. None of the 15 interaction tests for these features is significant (all Holm p > 0.8). The differences visible in the chart, such as the lower women's email lift in rural areas (+2.9 pp), are within what chance alone produces. Knowing a customer's recency, spend, channel, tenure or area changes neither which email we should send nor how much it helps.

**3. Buyer type is the only feature that changes the effect** (Holm p ≤ 0.0001 in all three comparisons):

| Buyer type | No-email visit rate | Men's email lift | Women's email lift | Men's vs women's | p (Holm), men's vs women's |
|---|---|---|---|---|---|
| Men's only | 10.0% | +6.9 pp | +1.1 pp | +5.8 pp | < 0.001 |
| Women's only | 9.6% | +7.1 pp | +7.4 pp | −0.3 pp | 0.53 |
| Both | 18.1% | +13.4 pp | +7.1 pp | +6.3 pp | < 0.001 |

- **The women's email only works on customers who buy women's items.** On men's-only buyers it adds almost nothing (+1.1 pp), while on women's-only buyers it is as good as the men's email.
- **The men's email works on everyone**, and it is never worse than the women's email.
- **"Both" buyers gain the most extra visits**: +13.4 pp from the men's email, almost twice as much as the other two groups. Part of this comes from their higher baseline (18.1%): in relative terms all three groups gain about 70 to 75%. For targeting, the absolute lift is what counts, because it is the number of extra visits per email sent.

**Why buyer type is the only segment worth using.** A segment is useful for targeting only if it changes a decision: *whom* to email, or *which* email to send. For recency, spend, channel, tenure and area, every segment gets the same answer (the men's email, with about the same lift), so splitting customers by them adds complexity without adding visits. Buyer type changes both decisions: it decides whether the women's email is a valid choice (yes for women's-only buyers, no for men's-only buyers), and it identifies the customers who gain the most from an email ("Both" buyers).

**What this means for the campaign:** sending the men's email to everyone is the simplest choice, and it is never worse than the alternative. The women's email is a valid option only for women's-only buyers. If the budget limits the number of emails, "Both" buyers should come first.

**Limits:**
- Each feature is analysed on its own. Features overlap (for example "Both" buyers may also be bigger spenders), so part of one feature's effect may come from another. Phase 4 models all the features together.
- Only `visit` is analysed: conversion and spend are too sparse to split into segments.
- These lifts are averages over segments, not predictions for individual customers. Those come from the uplift model in phase 4.